# Binary Fairness Smoke Test

Notebook de validacion para datasets binarios. Comprueba la inferencia de tipo de problema y que las metricas fairness marcadas como binarias no queden `SKIPPED` en un escenario binario.

In [1]:
!pwd

/home/antonio-marin-ortega/Escritorio/andromeda/andromeda-trust-lib_repo/notebooks


In [2]:
from __future__ import annotations

import importlib.util
import json
from pathlib import Path
import pickle
import sys
import types

import joblib
import pandas as pd

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists():
            return candidate
    raise RuntimeError("No se encontró la raíz del proyecto (carpeta que contiene 'src').")

ROOT = find_project_root(Path.cwd())

SRC_ROOT = ROOT / "src" / "trust_library"
DATA_DIR = ROOT / "models_and_data"
CONFIG_PATH = SRC_ROOT / "configs.json"


def register_package(name: str, path: Path) -> None:
    module = types.ModuleType(name)
    module.__path__ = [str(path)]
    sys.modules[name] = module


def load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


register_package("trust_library", SRC_ROOT)
register_package("trust_library.fairness", SRC_ROOT / "fairness")

load_module("trust_library.base_metric", SRC_ROOT / "base_metric.py")
load_module("trust_library.utils", SRC_ROOT / "utils.py")
load_module("trust_library.pillar", SRC_ROOT / "pillar.py")
load_module("trust_library.fairness.fairness_metrics_core", SRC_ROOT / "fairness" / "fairness_metrics_core.py")
load_module("trust_library.fairness.metrics", SRC_ROOT / "fairness" / "metrics.py")
fairness_module = load_module("trust_library.fairness.fairness", SRC_ROOT / "fairness" / "fairness.py")

sys.modules["trust_library.fairness"].FairnessPillar = fairness_module.FairnessPillar

from trust_library.base_metric import BaseMetric
from trust_library.utils import EvaluationContext, infer_problem_type

FairnessPillar = fairness_module.FairnessPillar



def load_model(path: Path):
    errors = []
    for loader_name, loader in (("pickle", lambda p: pickle.load(open(p, "rb"))), ("joblib", joblib.load)):
        try:
            return loader(path)
        except Exception as exc:
            errors.append(f"{loader_name}: {exc}")
    raise RuntimeError(f"No se pudo cargar {path.name}: {errors}")


def build_context(*, model, train_df: pd.DataFrame, test_df: pd.DataFrame, target_column: str, factsheet: dict, feature_columns: list[str] | None = None) -> EvaluationContext:
    if feature_columns is None:
        feature_columns = [column for column in train_df.columns if column != target_column]
    X_train = train_df[feature_columns]
    y_train = train_df[target_column].to_numpy().flatten()
    X_test = test_df[feature_columns]
    y_test = test_df[target_column].to_numpy().flatten()
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    y_prob_train = model.predict_proba(X_train) if hasattr(model, "predict_proba") else None
    y_prob_test = model.predict_proba(X_test) if hasattr(model, "predict_proba") else None
    is_classification, problem_type, class_labels = infer_problem_type(
        model=model,
        y_train=y_train,
        y_test=y_test,
        y_prob_train=y_prob_train,
        y_prob_test=y_prob_test,
    )
    return EvaluationContext(
        model=model,
        train_data=train_df,
        test_data=test_df,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        y_pred_train=y_pred_train,
        y_pred_test=y_pred_test,
        y_prob_train=y_prob_train,
        y_prob_test=y_prob_test,
        factsheet=factsheet,
        is_classification=is_classification,
        problem_type=problem_type,
        class_labels=class_labels,
    )


def assert_not_skipped(properties: dict, metric_key: str) -> None:
    metric_props = properties[metric_key]
    status = metric_props.get("Status")
    assert status != "SKIPPED", f"{metric_key} no deberia estar SKIPPED en binario: {metric_props}"


config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))


In [3]:
train_df = pd.read_csv(DATA_DIR / "fair_train.csv")
test_df = pd.read_csv(DATA_DIR / "fair_test.csv")
model = load_model(DATA_DIR / "model_fair.pkl")

factsheet = {
    "general": {
        "target_column": {"value": "income"},
        "model_type": {"value": type(model).__name__},
    },
    "fairness": {
        "protected_feature": {"value": "sex"},
        "protected_values": {"value": [1]},
        "favorable_outcomes": {"value": [1]},
    },
    "sustainability": {
        "use_codecarbon": {
            "value": True
        }
    }
}

context = build_context(
    model=model,
    train_df=train_df,
    test_df=test_df,
    target_column="income",
    factsheet=factsheet,
)

assert context.is_classification is True
assert context.problem_type == BaseMetric.ProblemType.BINARY
assert context.is_binary_classification is True
assert context.is_multiclass_classification is False
assert context.y_prob_positive is not None
assert list(context.class_labels) == [0, 1]

from trust_library.core import TrustEvaluator

evaluator = TrustEvaluator(
    model=model,
    train_data=train_df,
    test_data=test_df,
    factsheet=factsheet,
    config_path=CONFIG_PATH,  # o None si quieres la config por defecto
    output_path="trust_evaluation_result.json",
)

result = evaluator.evaluate(show_nan=True)

# Validar que ninguna métrica haya sido saltada en ningún pilar
for pillar_name, pillar_properties in result["properties"].items():
    for metric_key in pillar_properties.keys():
        assert_not_skipped(pillar_properties, metric_key)

# Resumen completo de estado de todas las métricas de todos los pilares
summary = {
    pillar_name: {
        metric_key: metric_properties.get("Status", "EXECUTED")
        for metric_key, metric_properties in pillar_properties.items()
    }
    for pillar_name, pillar_properties in result["properties"].items()
}

summary

[codecarbon WARNING @ 12:36:11] Multiple instances of codecarbon are allowed to run at the same time.


Building evaluation context...
Computing Fairness metrics...
Metric 'underfitting' computed in 0.00 seconds.
Metric 'overfitting' computed in 0.00 seconds.
Metric 'class_balance' computed in 0.00 seconds.
Metric 'statistical_parity_difference' computed in 0.00 seconds.
Metric 'disparate_impact' computed in 0.00 seconds.
Metric 'equal_opportunity_difference' computed in 0.00 seconds.
Metric 'average_odds_difference' computed in 0.00 seconds.
Metric 'accuracy_parity' computed in 0.00 seconds.
Metric 'predictive_parity' computed in 0.00 seconds.
Metric 'treatment_equality' computed in 0.00 seconds.
Metric 'calibration_gap' computed in 0.00 seconds.
Metric 'well_calibration_error' computed in 0.00 seconds.
Metric 'generalized_entropy_index' computed in 0.00 seconds.
Metric 'theil_index' computed in 0.00 seconds.
Metric 'coefficient_of_variation' computed in 0.00 seconds.
Metric 'individual_consistency' computed in 0.00 seconds.
Metric 'class_imbalance' computed in 0.00 seconds.
Metric 'kl_

{'fairness': {'underfitting': 'EXECUTED',
  'overfitting': 'EXECUTED',
  'class_balance': 'EXECUTED',
  'statistical_parity_difference': 'EXECUTED',
  'disparate_impact': 'EXECUTED',
  'equal_opportunity_difference': 'EXECUTED',
  'average_odds_difference': 'EXECUTED',
  'accuracy_parity': 'EXECUTED',
  'predictive_parity': 'EXECUTED',
  'treatment_equality': 'EXECUTED',
  'calibration_gap': 'EXECUTED',
  'well_calibration_error': 'EXECUTED',
  'generalized_entropy_index': 'EXECUTED',
  'theil_index': 'EXECUTED',
  'coefficient_of_variation': 'EXECUTED',
  'individual_consistency': 'EXECUTED',
  'class_imbalance': 'EXECUTED',
  'kl_divergence': 'EXECUTED',
  'conditional_demographic_disparity': 'EXECUTED',
  'smoothed_edf': 'EXECUTED',
  'bias_amplification': 'EXECUTED',
  'between_group_generalized_entropy_error': 'EXECUTED',
  'cohens_d': 'EXECUTED',
  'z_test_diff': 'EXECUTED'},
 'accountability': {'train_test_split': 'EXECUTED',
  'missing_data': 'EXECUTED',
  'normalization': 'EXE